# Import Needed Libraries

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains

import pandas as pd
import time
import random
import os
import re
from datetime import date, datetime
from uuid import uuid4

# Needed Cols

In [2]:
cities = [
    'Amsterdam',
    'Athens',
    'Barcelona',
    'Berlin',
    'Budapest',
    'Lisbon',
    'London',
    'Paris',
    'Rome',
    'Vienna'
]

# Scraping Cols data

# Listing ID (stable identifier extracted from the URL)

In [3]:
# The Airbnb room id inside the URL is the real identifier of a listing:
# the same listing can appear with different query strings, and names are not unique.
LISTING_ID_RE = re.compile(r"/rooms/(?:plus/)?(\d+)")

def extract_listing_id(url):
    """'https://www.airbnb.com/rooms/13952479?adults=4&...' -> '13952479'
    Kept as a string: ids can be 19 digits long (safe for CSV/Excel round-trips)."""
    m = LISTING_ID_RE.search(url or "")
    return m.group(1) if m else None

In [4]:
# --- ROBUST RATING/FAV EXTRACTOR ---
# Old bug: textContent glued the rating and the review count together ("4.81" + "719 reviews"
# -> "4.81719 reviews"), so the regex returned 81719 instead of 719 for Guest Favorite listings.
# Fix: join the text nodes with spaces, and make the regexes refuse digits that belong to a decimal.
RATING_RE  = re.compile(r"(?<![\d.])(\d\.\d{1,2})(?!\d)")
REVIEWS_RE = re.compile(r"(?<![\d.,])(\d[\d,]*)\s*reviews?", re.IGNORECASE)   # also handles "1,234 reviews"

TEXT_NODES_JS = """
const walker = document.createTreeWalker(arguments[0], NodeFilter.SHOW_TEXT);
const parts = [];
let node;
while ((node = walker.nextNode())) {
    const t = node.textContent.trim();
    if (t) parts.push(t);
}
return parts.join(' ');
"""

def spaced_text(driver, element):
    """Text of an element with a space between separate text nodes."""
    try:
        txt = driver.execute_script(TEXT_NODES_JS, element)
    except Exception:
        txt = element.get_attribute("textContent")
    return " ".join((txt or "").split())


def parse_review_block(clean_link, clean_parent):
    """Pure parsing logic -> (rating, reviews, is_fav)."""
    rating, reviews, is_fav = "N/A", "0", 0

    # --- SCENARIO A: GUEST FAVORITE ---
    if "Guest favorite" in clean_link or "Guest favorite" in clean_parent:
        is_fav = 1
        r_match = RATING_RE.search(clean_parent)
        if r_match: rating = r_match.group(1)
        rev_match = REVIEWS_RE.search(clean_parent)
        if rev_match: reviews = rev_match.group(1).replace(",", "")

    # --- SCENARIO B: STANDARD ---
    else:
        if "New" in clean_parent:
            rating = "New"
        else:
            r_match = RATING_RE.search(clean_parent)
            if r_match: rating = r_match.group(1)

        # Try link first, then parent for reviews
        rev_match = REVIEWS_RE.search(clean_link) or REVIEWS_RE.search(clean_parent)
        if rev_match: reviews = rev_match.group(1).replace(",", "")

    return rating, reviews, is_fav


def get_details_and_fav(driver, wait):
    rating, reviews, is_fav = "N/A", "0", 0

    try:
        # Locate the review anchor (link or button)
        try:
            review_link = wait.until(EC.presence_of_element_located(
                (By.XPATH, "//a[contains(@href, 'reviews')] | //button[contains(., 'reviews')]")
            ))
        except:
            # Fallback for "New" listings
            try:
                h1_parent = driver.find_element(By.TAG_NAME, "h1").find_element(By.XPATH, "./..").text
                if "New" in h1_parent: return "New", "0", 0
            except: pass
            return rating, reviews, is_fav

        clean_link = spaced_text(driver, review_link)
        # parent text (for standard listings where the rating is outside the link)
        clean_parent = spaced_text(driver, review_link.find_element(By.XPATH, "./.."))

        return parse_review_block(clean_link, clean_parent)

    except: pass
    return rating, reviews, is_fav

In [5]:
# --- PRICE EXTRACTOR WITH PRICE CONTEXT ---
# In this scraper, the displayed price represents the total for the selected stay.
# Price Basis is marked stay_total when a price is found and the stay duration is valid.
PRICE_RE = r"(?:E£|EGP|USD|EUR|[$€£])\s?\d[\d,\.]*"

def get_price_details(driver):
    """Return raw displayed price, currency, and the basis of the amount."""
    try:
        sidebar = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "[data-plugin-in-point-id='BOOK_IT_SIDEBAR']")
            )
        )
        time.sleep(1)
        text = " ".join(sidebar.get_attribute("textContent").split())
    except Exception:
        try:
            text = " ".join(
                driver.find_element(By.TAG_NAME, "body")
                .get_attribute("textContent").split()
            )
        except Exception:
            text = ""

    # Prefer an amount explicitly labelled as the stay total. If Airbnb's text
    # omits that label, use the displayed amount; this project treats it as the
    # total for the selected stay, as confirmed by the user.
    total_match = re.search(
        rf"({PRICE_RE})\s*total", text, re.IGNORECASE
    )
    if total_match:
        raw_price = total_match.group(1)
    else:
        price_match = re.search(PRICE_RE, text, re.IGNORECASE)
        raw_price = price_match.group(0) if price_match else None

    basis = (
        "stay_total"
        if raw_price and pd.to_numeric(STAY_NIGHTS, errors="coerce") > 0
        else ("displayed_price_unknown_basis" if raw_price else "not_found")
    )

    currency = None
    if raw_price:
        currency_match = re.match(
            r"\s*(E£|EGP|USD|EUR|[$€£])", raw_price, re.IGNORECASE
        )
        if currency_match:
            symbol = currency_match.group(1).upper()
            currency_map = {
                "$": "USD", "€": "EUR", "£": "GBP",
                "E£": "EGP", "EGP": "EGP", "USD": "USD", "EUR": "EUR"
            }
            currency = currency_map.get(symbol, symbol)

    return {
        "raw": raw_price,
        "currency": currency,
        "basis": basis,
    }

In [6]:
all_data = []
city_count = 0
seen_ids = set()      # listings actually saved (across ALL cities/pages)
skipped_ids = set()   # listings skipped (no rating) -> never reopened on later pages/cities

class SkipListing(Exception):
    """Listing not wanted (e.g. no numeric rating): skipped without counting as a failure."""
    pass

def is_valid_rating(rating):
    val = pd.to_numeric(rating, errors="coerce")   # "N/A" and "New" become NaN
    return pd.notna(val) and 0 < val <= 5

# Distinguish when the data was scraped from the dates used to search for a stay.
SCRAPE_DATE = date.today().isoformat()
SCRAPE_TIMESTAMP = datetime.now().astimezone().isoformat(timespec="seconds")
SCRAPE_RUN_ID = str(uuid4())

# Search parameters are recorded in every row so future reports can interpret prices.
STAY_CHECKIN = "2027-02-01"
STAY_CHECKOUT = "2027-02-06"
GUESTS = 4
ROOM_TYPE_FILTER = "Entire home/apt"
MINIMUM_BEDROOMS = 2
STAY_NIGHTS = (
    date.fromisoformat(STAY_CHECKOUT) - date.fromisoformat(STAY_CHECKIN)
).days

for d in ("metadata/images", "metadata/webpages", "scraped_dataset"):
    os.makedirs(d, exist_ok=True)

# Loop of Scraping

In [7]:
for city in cities:
    city_count += 1

    URL = (f"https://www.airbnb.com/s/{city}/homes?"
           "room_types%5B%5D=Entire%20home%2Fapt"
           f"&min_bedrooms={MINIMUM_BEDROOMS}"
           f"&checkin={STAY_CHECKIN}&checkout={STAY_CHECKOUT}"
           f"&adults={GUESTS}&currency=EUR")

    chrome_options = Options()
    chrome_options.binary_location = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36")

    driver = webdriver.Chrome(options=chrome_options)

    target_count = 50
    items_scraped_this_city = 0

    try:
        driver.get(URL)
        wait = WebDriverWait(driver, 20)
        print(f"\n=== SCRAPING CITY {city_count}/{len(cities)}: {city} ===\n")
        print("Page title:", driver.title, "| URL:", driver.current_url)

        # --- Retry the first page up to 3 times (handles one-off timeouts) ---
        cards_ok = False
        for attempt in range(1, 4):
            try:
                wait.until(EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[data-testid='card-container']")))
                cards_ok = True
                break
            except Exception as e:
                print(f"No cards (attempt {attempt}/3) ->", type(e).__name__)
                driver.save_screenshot(f"metadata/images/debug_{city}_{attempt}.png")
                if attempt < 3:
                    driver.refresh()
        if not cards_ok:
            print("Giving up on", city)
            continue   # 'finally' still runs and quits the driver

        while items_scraped_this_city < target_count:
            print(f"--- Scraping Page (Collected: {len(all_data)}) ---")

            try:
                wait.until(EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[data-testid='card-container']")))
            except Exception as e:
                print("NO CARDS FOUND ->", type(e).__name__)
                driver.save_screenshot(f"metadata/images/debug_{city}.png")
                break

            listings = driver.find_elements(By.CSS_SELECTOR, "div[data-testid='card-container']")
            print("Cards found:", len(listings))

            listing_urls = []          # list of (listing_id, url), unique ids only
            page_ids = set()
            for l in listings:
                try:
                    href = l.find_element(By.TAG_NAME, "a").get_attribute("href")
                except Exception as e:
                    print("No link in card ->", type(e).__name__)
                    continue
                lid = extract_listing_id(href)
                if not lid:
                    print("No listing_id in URL ->", (href or "")[:60])
                    continue
                if lid in seen_ids or lid in skipped_ids or lid in page_ids:
                    continue           # already scraped / repeated card
                page_ids.add(lid)
                listing_urls.append((lid, href))
            print("New listings on this page:", len(listing_urls))

            main_window = driver.current_window_handle
            fail_streak = 0

            for listing_id, url in listing_urls:
                if items_scraped_this_city >= target_count:
                    break

                try:
                    driver.execute_script(f"window.open('{url}', '_blank');")
                    driver.switch_to.window(driver.window_handles[-1])
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

                    # Close any popup (translation / cookies)
                    ActionChains(driver).send_keys(Keys.ESCAPE).perform()

                    # A. Name
                    try:
                        name = driver.find_element(By.TAG_NAME, "h1").text
                    except Exception:
                        name = "No Title"

                    # B. Superhost
                    is_superhost = "No"
                    try:
                        driver.execute_script("window.scrollBy(0, 500);")
                        time.sleep(0.5)
                        body_text = driver.find_element(By.TAG_NAME, "body").text
                        if "Superhost" in body_text:
                            is_superhost = "Yes"
                    except Exception:
                        pass

                    # C. Rating/Fav (checked first: no rating -> skip before waiting for the price)
                    rating, reviews, is_fav = get_details_and_fav(driver, wait)
                    if not is_valid_rating(rating):
                        raise SkipListing(f"no numeric rating ({rating})")

                    # D. Price (listing is saved only if BOTH rating and price exist)
                    price_info = get_price_details(driver)
                    price = price_info["raw"]
                    if not price:
                        raise ValueError("price not found")

                    all_data.append({
                        "listing_id": listing_id,
                        "Property Name": name,
                        "Price": price,
                        "Rating": rating,
                        "Number of Reviews": reviews,
                        "Superhost Status": is_superhost,
                        "Guest Favorite Status": "Yes" if is_fav else "No",
                        "Property URL": url,
                        "City": city,  # Kept for backward compatibility; this is the search city.
                        "Scraped At": SCRAPE_DATE,
                        "Search City": city,
                        "Price Raw": price_info["raw"],
                        "Price Currency": price_info["currency"],
                        "Price Basis": price_info["basis"],
                        "Stay Check-in": STAY_CHECKIN,
                        "Stay Check-out": STAY_CHECKOUT,
                        "Stay Nights": STAY_NIGHTS,
                        "Adults": GUESTS,
                        "Room Type Filter": ROOM_TYPE_FILTER,
                        "Minimum Bedrooms": MINIMUM_BEDROOMS,
                        "Scraped At Timestamp": SCRAPE_TIMESTAMP,
                        "Scrape Run ID": SCRAPE_RUN_ID,
                    })
                    seen_ids.add(listing_id)

                    print(
                        f"[{len(all_data):03}] "
                        f"ID: {listing_id} | "
                        f"{city} | "
                        f"{price} | "
                        f"Rating: {rating} ({reviews} reviews) | "
                        f"Favorite: {is_fav} | "
                        f"Superhost: {is_superhost}"
                    )
                    
                    time.sleep(random.uniform(1.0, 2.0))

                    driver.close()
                    driver.switch_to.window(main_window)

                    items_scraped_this_city += 1
                    fail_streak = 0

                except SkipListing as e:
                    print(f"SKIPPED [{listing_id}] -> {e}")
                    skipped_ids.add(listing_id)
                    if len(driver.window_handles) > 1:
                        driver.close()
                        driver.switch_to.window(main_window)
                    continue   # does not increase items_scraped nor fail_streak

                except Exception as e:
                    print("LISTING FAILED:", url[:60], "->", type(e).__name__, str(e)[:150])

                    # Save one debug snapshot (listing tab must still be open)
                    if not os.path.exists("metadata/webpages/debug_listing.html"):
                        try:
                            with open("metadata/webpages/debug_listing.html", "w", encoding="utf-8") as f:
                                f.write(driver.page_source)
                            driver.save_screenshot("metadata/images/debug_listing.png")
                        except Exception:
                            pass

                    fail_streak += 1
                    if len(driver.window_handles) > 1:
                        driver.close()
                        driver.switch_to.window(main_window)

                    if fail_streak >= 5:
                        print("5 failures in a row, stopping city")
                        items_scraped_this_city = target_count
                        break
                    continue

            if items_scraped_this_city < target_count:
                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, "a[aria-label='Next']")
                    driver.execute_script("arguments[0].click();", next_btn)
                    time.sleep(4)
                except Exception as e:
                    print("No Next button ->", type(e).__name__)
                    break

    finally:
        print(f"\nCompleted! Total scraped: {len(all_data)}")
        driver.quit()


=== SCRAPING CITY 1/10: Amsterdam ===

Page title: Airbnb | Amsterdam - Vacation Rentals & Places to Stay - North Holland | URL: https://www.airbnb.com/s/Amsterdam/homes?room_types%5B%5D=Entire%20home%2Fapt&min_bedrooms=2&checkin=2027-02-01&checkout=2027-02-06&adults=4&currency=EUR
No cards (attempt 1/3) -> TimeoutException
--- Scraping Page (Collected: 0) ---
Cards found: 24
New listings on this page: 24
[001] ID: 13952479 | Amsterdam | €1,446 | Rating: 4.81 (719 reviews) | Favorite: 1 | Superhost: No
[002] ID: 22463309 | Amsterdam | €1,154 | Rating: 4.89 (201 reviews) | Favorite: 1 | Superhost: Yes
[003] ID: 703633681123945658 | Amsterdam | €2,112 | Rating: 4.99 (245 reviews) | Favorite: 1 | Superhost: Yes
[004] ID: 1113044804402014015 | Amsterdam | €1,379 | Rating: 4.56 (235 reviews) | Favorite: 0 | Superhost: No
[005] ID: 30762441 | Amsterdam | €1,469 | Rating: 4.81 (1775 reviews) | Favorite: 0 | Superhost: Yes
[006] ID: 1147691297463976343 | Amsterdam | €1,003 | Rating: 4.98 (92 

# Assign Scraped_Data in DataFrame

In [8]:
cols = [
    # Listing identity and location
    "listing_id", "Property Name", "Property URL", "City", "Search City",

    # Price information
    "Price", "Price Raw", "Price Currency", "Price Basis",
    "Estimated Price Per Night",

    # Ratings and host/property status
    "Rating", "Number of Reviews", "Superhost Status", "Guest Favorite Status",

    # Search / stay context
    "Stay Check-in", "Stay Check-out", "Stay Nights", "Adults",
    "Room Type Filter", "Minimum Bedrooms",

    # Collection metadata
    "Scraped At", "Scraped At Timestamp", "Scrape Run ID",
]

df_scraped = pd.DataFrame(all_data)

# Make sure the notebook still produces a valid, structured output if no listings were collected.
for col in cols:
    if col not in df_scraped.columns:
        df_scraped[col] = pd.Series(dtype="object")
df_scraped = df_scraped[cols]

# listing_id is text: IDs can be long and must not be converted to floating point.
df_scraped["listing_id"] = df_scraped["listing_id"].astype("string")
df_scraped = df_scraped.drop_duplicates(
    subset="listing_id", keep="first"
).reset_index(drop=True)

# Parse numeric fields while preserving the original displayed amount in `Price Raw`.
df_scraped["Rating"] = pd.to_numeric(df_scraped["Rating"], errors="coerce")
# Safety net: never keep a listing without a numeric rating
df_scraped = df_scraped.dropna(subset=["Rating"]).reset_index(drop=True)
df_scraped["Number of Reviews"] = pd.to_numeric(
    df_scraped["Number of Reviews"], errors="coerce"
).astype("Int64")

def parse_price_amount(value):
    """Convert a currency-formatted amount to a number without changing Price Raw."""
    if pd.isna(value):
        return float("nan")
    amount = re.sub(r"[^0-9,\.]", "", str(value)).strip()
    if not amount:
        return float("nan")

    # Airbnb price strings commonly use commas as thousands separators (e.g. 1,234.56).
    # If both separators exist, the last separator is treated as the decimal separator.
    if "," in amount and "." in amount:
        if amount.rfind(",") > amount.rfind("."):
            amount = amount.replace(".", "").replace(",", ".")
        else:
            amount = amount.replace(",", "")
    elif "," in amount:
        tail = amount.rsplit(",", 1)[-1]
        if len(tail) in (1, 2):
            amount = amount.replace(",", ".")
        else:
            amount = amount.replace(",", "")
    return pd.to_numeric(amount, errors="coerce")

df_scraped["Price"] = df_scraped["Price Raw"].apply(parse_price_amount)

# The displayed Price is the total for the selected stay, so estimate the nightly
# price by dividing by Stay Nights. Keep the result null if price/nights are invalid.
df_scraped["Stay Nights"] = pd.to_numeric(
    df_scraped["Stay Nights"], errors="coerce"
)
df_scraped["Estimated Price Per Night"] = (
    df_scraped["Price"] / df_scraped["Stay Nights"]
).where(
    df_scraped["Price"].notna() & df_scraped["Stay Nights"].gt(0)
)

# Record the price basis consistently with the confirmed scraper behavior.
valid_stay_total = (
    df_scraped["Price"].notna() & df_scraped["Stay Nights"].gt(0)
)
df_scraped.loc[valid_stay_total, "Price Basis"] = "stay_total"
df_scraped.loc[~valid_stay_total & df_scraped["Price"].notna(), "Price Basis"] = (
    "displayed_price_unknown_basis"
)

# Helpful quality checks for the notebook/report.
assert df_scraped["listing_id"].is_unique
assert df_scraped["listing_id"].notna().all()
assert df_scraped["Rating"].notna().all()
print(f"Rows collected: {len(df_scraped):,}")
print(f"Unique listings: {df_scraped['listing_id'].nunique():,}")
print(f"Missing numeric prices: {df_scraped['Price'].isna().sum():,}")
print(f"Rows with stay-total price: {df_scraped['Price Basis'].eq('stay_total').sum():,}")
print(f"Missing estimated nightly prices: {df_scraped['Estimated Price Per Night'].isna().sum():,}")
print(f"Scrape run ID: {SCRAPE_RUN_ID}")

df_scraped

Rows collected: 500
Unique listings: 500
Missing numeric prices: 0
Rows with stay-total price: 500
Missing estimated nightly prices: 0
Scrape run ID: eeb856ec-b43c-4669-aa4c-31208e2a26f7


,listing_id,Property Name,Property URL,City,Search City,Price,Price Raw,Price Currency,Price Basis,Estimated Price Per Night,...,Guest Favorite Status,Stay Check-in,Stay Check-out,Stay Nights,Adults,Room Type Filter,Minimum Bedrooms,Scraped At,Scraped At Timestamp,Scrape Run ID
0,13952479,"Amazing Canal Penthouse, Stunning views",https://www.airbnb.com/rooms/13952479?adults=4...,Amsterdam,Amsterdam,1446,"€1,446",EUR,stay_total,289.2,...,Yes,2027-02-01,2027-02-06,5,4,Entire home/apt,2,2026-10-28,2026-10-28T13:05:11+03:00,eeb856ec-b43c-4669-aa4c-31208e2a26f7
1,22463309,"Very nice appartment,close to metro, free Park...",https://www.airbnb.com/rooms/22463309?adults=4...,Amsterdam,Amsterdam,1154,"€1,154",EUR,stay_total,230.8,...,Yes,2027-02-01,2027-02-06,5,4,Entire home/apt,2,2026-10-28,2026-10-28T13:05:11+03:00,eeb856ec-b43c-4669-aa4c-31208e2a26f7
2,703633681123945658,3 BEDRM APP (90m2) with canalview near Vondelpark,https://www.airbnb.com/rooms/70363368112394565...,Amsterdam,Amsterdam,2112,"€2,112",EUR,stay_total,422.4,...,Yes,2027-02-01,2027-02-06,5,4,Entire home/apt,2,2026-10-28,2026-10-28T13:05:11+03:00,eeb856ec-b43c-4669-aa4c-31208e2a26f7
3,1113044804402014015,Numa | 2 Bedroom Apartment in Amsterdam East,https://www.airbnb.com/rooms/11130448044020140...,Amsterdam,Amsterdam,1379,"€1,379",EUR,stay_total,275.8,...,No,2027-02-01,2027-02-06,5,4,Entire home/apt,2,2026-10-28,2026-10-28T13:05:11+03:00,eeb856ec-b43c-4669-aa4c-31208e2a26f7
4,30762441,Stylish Serviced Apartment near Central Station,https://www.airbnb.com/rooms/30762441?adults=4...,Amsterdam,Amsterdam,1469,"€1,469",EUR,stay_total,293.8,...,No,2027-02-01,2027-02-06,5,4,Entire home/apt,2,2026-10-28,2026-10-28T13:05:11+03:00,eeb856ec-b43c-4669-aa4c-31208e2a26f7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,43683531,Skyflats Vienna West View,https://www.airbnb.com/rooms/43683531?adults=4...,Vienna,Vienna,1445,"€1,445",EUR,stay_total,289.0,...,Yes,2027-02-01,2027-02-06,5,4,Entire home/apt,2,2026-10-28,2026-10-28T13:05:11+03:00,eeb856ec-b43c-4669-aa4c-31208e2a26f7
496,1315168591643009839,TOP 11 | 5 rooms | 100 m² | Schönbrunn Palace,https://www.airbnb.com/rooms/13151685916430098...,Vienna,Vienna,1232,"€1,232",EUR,stay_total,246.4,...,Yes,2027-02-01,2027-02-06,5,4,Entire home/apt,2,2026-10-28,2026-10-28T13:05:11+03:00,eeb856ec-b43c-4669-aa4c-31208e2a26f7
497,1217561415505762121,"Vienna Residence, Prater - Exhibition Center #...",https://www.airbnb.com/rooms/12175614155057621...,Vienna,Vienna,1063,"€1,063",EUR,stay_total,212.6,...,No,2027-02-01,2027-02-06,5,4,Entire home/apt,2,2026-10-28,2026-10-28T13:05:11+03:00,eeb856ec-b43c-4669-aa4c-31208e2a26f7
498,1634646938982765723,Central apartment in Viennese old building,https://www.airbnb.com/rooms/16346469389827657...,Vienna,Vienna,182,€182,EUR,stay_total,36.4,...,Yes,2027-02-01,2027-02-06,5,4,Entire home/apt,2,2026-10-28,2026-10-28T13:05:11+03:00,eeb856ec-b43c-4669-aa4c-31208e2a26f7


# Save Dataset as CSV file

In [9]:
# Keep listing IDs as text when reading the CSV back (some IDs can be very long).
# All original fields are retained alongside additional reporting fields.
output_path = "scraped_dataset/Airbnb_Scraped_Dataset.csv"
df_scraped.to_csv(output_path, index=False)
print(f"Saved {len(df_scraped):,} rows and {len(df_scraped.columns):,} columns to: {output_path}")

Saved 500 rows and 23 columns to: scraped_dataset/Airbnb_Scraped_Dataset.csv
